## Disaster Tweets Analysis
The following notebook is a quick analysis regarding natural language processing, the main goal of this notebook is to predict wether a particular tweet is of a natrual disaster content or not. 
The following techinques are used to solve the problem 
- Data cleaning 
- Data processing 
- Creating CNN Model

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

import nltk
nltk.download('punkt')
import os
import re
import io
import string
import pandas as pd
from nltk import word_tokenize
from nltk.tokenize import sent_tokenize
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models
from keras.preprocessing import sequence
from keras.preprocessing import text
from keras.models import Sequential
from keras.layers import  Activation
from tensorflow.keras.preprocessing.sequence import pad_sequences
from pickle import load
from keras.preprocessing.text import Tokenizer
from keras.utils.vis_utils import plot_model
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Dropout
from keras.layers import Embedding
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from keras.layers.merge import concatenate




## 1- Loading the training and testing datasets 
Loading the data using the pandas.read_csv() function 

In [ ]:
training_data = pd.read_csv('../input/nlp-getting-started/train.csv')
testing_data = pd.read_csv('../input/nlp-getting-started/train.csv')

training_data.head(5)

# 2- Creating train/test sets 


In [ ]:
train_X = training_data['text']
train_Y = training_data['target'] 

test_X = testing_data['text']


## 3-Cleaning the data 
Data cleaning is something inevetable when working with text data, the cleaning task in done by doing the following tasks: 
- remove punctuations  
- replace HTML tags with a space 
- replace all found emoticons in the text with the replacement char
- lowercasing the text 
- changing each number occurance with # symbol 
- keeping the emoticons in their right place for semantic anaylsis 


In [ ]:
def preprocess(raw_text):
    #regex found in: https://stackoverflow.com/questions/28077049/regex-matching-emoticons
    smiley_regex = '(:\w+:|<[/\]?3|[()\\D|*$][-^]?[:;=]|[:;=B8][-^]?[3DPp@$*\)(/|])(?=\s|[!.?]|$)'
    #replacement character for emoticons
    replace_char_regex = '§'
    #takes raw text and replaces <br/> with a space
    raw_text = raw_text.replace('<br />',' ')
    #find all emoticons, defined in smiley_regex and save them in variable emoticons
    emoticons = re.findall(smiley_regex, raw_text)
    #replace all found emoticons in the text with the replacement char, defined in replace_char_regex
    text_replaces = re.sub(smiley_regex, replace_char_regex , raw_text)
    
    #make all text symbols lower case
    lower_text = text_replaces.lower()
    #removes all punctuations from the text    
    entry_no_punct = lower_text.translate(str.maketrans('', '', string.punctuation))
    
    #replace the replace_char_regex with the original emoticons again on the right possitions in text
    for emoticon in emoticons:
        entry_no_punct = entry_no_punct.replace(replace_char_regex, emoticon, 1)

    #replace all digits with '#'
    entry_hash_num = ''.join([s if not s.isdigit() else '#' for s in entry_no_punct])

    return entry_hash_num



training_data['text'] = training_data['text'].apply(lambda x: preprocess(x))
testing_data['text'] = testing_data['text'].apply(lambda x: preprocess(x))

train_X = training_data['text']
train_Y = training_data['target'] 

test_X = testing_data['text']

## Preprocessing the data 
For our data to be passed to out model, some preprocesing steps are needed which are the following: 
- tokenize the text 
- cut-pad of the input text to a max length as needed which in this case it was 350 works 

In [ ]:
tokenizer = text.Tokenizer(num_words=10000)
tokenizer.fit_on_texts(train_X)

# converting each dataset into a vector of numbers 
train_set = tokenizer.texts_to_sequences(train_X)
test_set = tokenizer.texts_to_sequences(test_X)

# converting each labels subset into a numpy array
train_labels = train_Y.to_numpy()


# making our data consistent and un changble by making the vectors have a fixed max length
train_data = keras.preprocessing.sequence.pad_sequences(train_set, maxlen=350)
test_data = keras.preprocessing.sequence.pad_sequences(test_set, maxlen=350)

## Creating the model
for this competition i used CNN neural networks, which are usaally used in image classification, but as it's gonna turn out, it can be used for sentiment analysis and machine learning text classification tasks.
the vanilla model contains the folowing layers:
- input layer 
- embedding layer 
- convolutional layer with 1D (in our case it's text data and its 1 dimensional)
- max pooling layer
- flatten layer
- 2 dense layeres 


In [ ]:
def define_vanilla_model(length, vocab_size):
    input=Input(shape=(length,))
    embedding=Embedding(vocab_size, 150)(input)
    conv = Conv1D(filters=32, kernel_size=3, activation='relu')(embedding)
    pool = MaxPooling1D(pool_size=3)(conv)
    flatten = Flatten()(pool)
    dense1 = Dense(10, activation='relu')(flatten)
    
    #second dense layer to get a accuracy representation between 0 and 1
    out = Dense(1, activation='sigmoid')(dense1)

    model = Model(input, outputs=out)
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    print(model.summary())
    plot_model(model, show_shapes=True, to_file='define_model.png')	
    return model    

## Fitting the model 

In [ ]:
vanilla_model = define_vanilla_model(350,10000)
vanilla_model.fit(train_data, train_labels, epochs=10, batch_size=16)


## Predictions 
Basically the prebuilt predict() function predicts the probability of a tweet being about a natural disaster or not, these probabilities are in domain  [0,1] and therefore i constructed i constructed the predictions function which takes the predicted test_Y and count every tweet that has a probability higher or equal to 70% to be a natrual disaster tweet while the rest are not.

In [ ]:
test_Y = vanilla_model.predict(test_data)

In [ ]:
def predictions(tweets):
    predict = []
    temp_tweets = np.array(tweets)
    for i in range(len(tweets)):
        if temp_tweets[i] >= 0.7:
            predict.append(1)
        else:
            predict.append(0)
    return predict

In [ ]:
import seaborn as sns
predict = predictions(test_Y)
output = pd.DataFrame({'id': testing_data.id, 'target': predict})
output.to_csv('submission.csv', index=False)

In [ ]:
plotting = pd.DataFrame(predict,columns=["target"])
sns.countplot(x="target",data=plotting)